# Document Processing -- From Raw Files to Chunks

## What You Will Learn

In any Retrieval Augmented Generation (RAG) system, the very first step is **document processing**: taking raw files (PDFs, Markdown, plain text) and transforming them into small, focused passages called **chunks** that can be embedded and searched.

This notebook walks through the chunking stage of the pipeline using two strategies from `agentexplorr.rag.chunking`:

1. **FixedSizeChunker** -- splits text into fixed-character windows with overlap.
2. **RecursiveChunker** -- splits on a hierarchy of separators (paragraphs, lines, sentences, words) to preserve natural document structure.

## Prerequisites

* Python 3.10+
* The `agentexplorr` package installed (`pip install -e .` from the repo root)
* No external API keys or services required -- everything runs locally.

## Pipeline Overview

```
Raw File  -->  DocumentProcessor  -->  Chunker  -->  [Chunk, Chunk, ...]
                  (load & parse)       (split)         ready for embedding
```

In [ ]:
from agentexplorr.rag.chunking import (
    Chunk,
    FixedSizeChunker,
    RecursiveChunker,
)

print("Imports successful!")
print(f"  Chunk:             {Chunk}")
print(f"  FixedSizeChunker:  {FixedSizeChunker}")
print(f"  RecursiveChunker:  {RecursiveChunker}")

## Why Chunking Matters for RAG Quality

Chunking is the **single most overlooked factor** in RAG quality. Bad chunks lead to bad retrieval which leads to bad answers.

Embedding models have a finite context window (typically 256-512 tokens for sentence-transformers). Even if a model *could* embed an entire book, the resulting single vector would be a blurry average of every topic -- useless for precise retrieval.

Good chunks should be:

| Property | Why It Matters |
|---|---|
| **Self-contained** | The chunk makes sense on its own, without needing surrounding context. |
| **Focused** | Each chunk covers one idea so the retriever can pinpoint the right passage. |
| **Right-sized** | Small enough to be specific, large enough to be meaningful (typically 200-1000 chars). |

Let's see both strategies in action on the same sample text.

In [ ]:
# --- Sample text: a short article with clear paragraph structure ---
sample_text = """Transformers are a neural network architecture introduced in the 2017 paper
"Attention Is All You Need" by Vaswani et al. They replaced recurrent layers with
self-attention mechanisms, enabling much greater parallelism during training.

The key innovation is the self-attention mechanism. For each token in the input,
attention computes a weighted sum over all other tokens. The weights are learned,
allowing the model to focus on the most relevant context regardless of distance.

Transformers have become the backbone of modern NLP. Models like BERT, GPT, and T5
are all based on the Transformer architecture. They achieve state-of-the-art results
on tasks ranging from translation to question answering to code generation.

More recently, Transformers have expanded beyond text. Vision Transformers (ViT)
apply the same self-attention mechanism to image patches, achieving competitive
results with convolutional networks on image classification benchmarks."""

print(f"Sample text length: {len(sample_text)} characters\n")

# --- Fixed-size chunking with chunk_size=200 and overlap=30 ---
fixed_chunker = FixedSizeChunker(chunk_size=200, overlap=30)
fixed_chunks = fixed_chunker.chunk(sample_text, metadata={"source": "transformers_article"})

print(f"FixedSizeChunker produced {len(fixed_chunks)} chunks (size=200, overlap=30)\n")
for i, c in enumerate(fixed_chunks):
    print(f"--- Chunk {i} ({len(c.text)} chars) ---")
    print(c.text)
    print()

## Fixed-Size vs Recursive Chunking

Notice the problem above: the `FixedSizeChunker` slices the text every 200 characters regardless of where paragraphs or sentences begin or end. A chunk might start in the middle of a sentence about attention and end halfway through a sentence about BERT.

The **RecursiveChunker** solves this by trying separators in order of strength:

1. `\n\n` -- paragraph breaks (strongest semantic boundary)
2. `\n` -- line breaks
3. `. ` -- sentence endings
4. ` ` -- word boundaries
5. `""` -- character-level (emergency fallback)

If a piece is still too large after splitting on the current separator, it recurses with the next (weaker) one. After splitting, it **merges** small consecutive pieces back together to avoid tiny chunks.

The result: chunks that respect natural document structure while staying within the size limit.

In [ ]:
# --- Recursive chunking with the same size parameters ---
recursive_chunker = RecursiveChunker(chunk_size=200, overlap=30)
recursive_chunks = recursive_chunker.chunk(sample_text, metadata={"source": "transformers_article"})

print(f"RecursiveChunker produced {len(recursive_chunks)} chunks (size=200, overlap=30)\n")
for i, c in enumerate(recursive_chunks):
    print(f"--- Chunk {i} ({len(c.text)} chars) ---")
    print(c.text)
    print()

# --- Side-by-side comparison ---
print("=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Metric':<30} {'Fixed':>10} {'Recursive':>10}")
print("-" * 60)
print(f"{'Number of chunks':<30} {len(fixed_chunks):>10} {len(recursive_chunks):>10}")
print(f"{'Avg chunk length (chars)':<30} "
      f"{sum(len(c.text) for c in fixed_chunks) // max(len(fixed_chunks), 1):>10} "
      f"{sum(len(c.text) for c in recursive_chunks) // max(len(recursive_chunks), 1):>10}")
print(f"{'Min chunk length':<30} "
      f"{min(len(c.text) for c in fixed_chunks):>10} "
      f"{min(len(c.text) for c in recursive_chunks):>10}")
print(f"{'Max chunk length':<30} "
      f"{max(len(c.text) for c in fixed_chunks):>10} "
      f"{max(len(c.text) for c in recursive_chunks):>10}")

# Show that recursive chunks tend to align with paragraph boundaries
print("\n--- Do recursive chunks start at paragraph boundaries? ---")
for i, c in enumerate(recursive_chunks):
    first_words = c.text.strip().split()[:5]
    print(f"  Chunk {i} starts with: \"{' '.join(first_words)}...\"")

## Choosing Chunk Size and Overlap

There is no universally optimal chunk size -- it depends on your data and your embedding model. Here are practical guidelines:

**Chunk Size**

| Range | Best For |
|---|---|
| 100-300 chars | Short, precise answers (FAQ-style). Higher precision, lower recall. |
| 300-800 chars | General-purpose RAG. Good balance of context and specificity. |
| 800-1500 chars | Long-form answers or documents with long paragraphs. |

**Overlap**

* **0%** -- No overlap. Fastest, but sentences at boundaries may be lost.
* **10-20%** of chunk size -- Typical production setting. Ensures boundary sentences appear in both adjacent chunks.
* **>30%** -- Diminishing returns; creates many near-duplicate chunks that waste storage and slow search.

**Rules of Thumb**

1. Start with `RecursiveChunker(chunk_size=500, overlap=50)` and measure retrieval quality.
2. If answers are too vague, reduce chunk size for more focused passages.
3. If answers lack context, increase chunk size to give the LLM more to work with.
4. Always evaluate with real queries -- there is no substitute for empirical testing.

## Key Takeaways

1. **Chunking is the foundation of RAG quality.** The best embedding model and the best LLM cannot compensate for poorly chunked documents.

2. **FixedSizeChunker** is simple and predictable but ignores document structure. Use it as a baseline or for structureless text like raw logs.

3. **RecursiveChunker** respects paragraph and sentence boundaries and is the recommended default for most RAG applications. It is the strategy used by LangChain's `RecursiveCharacterTextSplitter`.

4. **Overlap** acts as insurance against boundary effects. A 10-20% overlap is a practical starting point.

5. Each `Chunk` object carries a **deterministic `chunk_id`** (SHA-256 hash of the text) which makes deduplication trivial when re-processing documents.

## Next Steps

* **02_vector_stores.ipynb** -- Learn how chunks are embedded into vectors and stored for similarity search.
* **03_rag_pipeline.ipynb** -- See the full end-to-end RAG pipeline that wires chunking, embedding, retrieval, and generation together.
* Explore `SemanticChunker` in `agentexplorr.rag.chunking` for topic-aware chunking using embedding similarity (requires `sentence-transformers`).